[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/lab07_what_care_costs.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Lab 7 — <your last name>` (click the title at the top left).

**To submit:** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Lab 7 — What care costs: utilization and prices
*Week 7. Baseline: sections 1–5. Powerups: section 6.*

What does a hospital stay cost, and who decides? This week has two halves. First, **Medicare utilization data**:
what every hospital in the country billed and was paid for one common admission, heart failure. Second,
**price transparency**: the files hospitals must now publish with the rates they negotiated with each insurer.
The question in both halves is the same: how much does the price of the *same* service vary, and why?

**How labs work.** Save a copy in Drive, work in your copy, share it back with the instructor
when you submit. Every lab has a **baseline** (everyone finishes it in the session) and
**powerups** (optional depth). AI assistants may write code for you; **you own every line** and
may be asked to explain any of it. Cells marked `# TODO` are yours to fill in.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


## 1. One admission, every hospital: Medicare inpatient data
CMS publishes **Medicare Inpatient Hospitals — by Provider and Service** every year: one row per hospital × DRG
(diagnosis-related group, the payment category for an inpatient stay) for **Original Medicare** (fee-for-service,
Part A) stays at hospitals paid under the inpatient prospective payment system (IPPS). The current release covers
**calendar 2024**. Medicare Advantage stays are not in it. A hospital × DRG cell with fewer than 11 discharges is
not published, so small hospitals drop out for rare DRGs.

Three money columns, and the lab turns on the difference between them:

| Column | Our name | What it is |
|---|---|---|
| `Avg_Submtd_Cvrd_Chrg` | `charges` | what the hospital **billed** (its list price, from its chargemaster) |
| `Avg_Tot_Pymt_Amt` | `total_pay` | what the hospital was actually **paid**: Medicare's payment plus the patient's deductible and coinsurance and any other primary payer |
| `Avg_Mdcr_Pymt_Amt` | `medicare_pay` | Medicare's share of `total_pay` |

All three are **averages per discharge** at that hospital. The full file has ~146,000 rows, so the loader below asks
the CMS data API for one DRG only. We use **DRG 291, heart failure and shock with major complications**, the
condition you have followed since week 2.

**Baseline:** report how many hospitals and discharges are in the file for DRG 291. Then, for `total_pay` and
`charges`, report the 10th, 50th and 90th percentiles across hospitals and the **p90/p10 ratio**. One sentence:
which varies more across hospitals, and why would you expect that?

In [ ]:
# The CMS data API (data.cms.gov/data-api) serves this dataset 5,000 rows at a time and can filter
# server-side, so we download only the DRG we need (~2,600 rows, about 1 MB).
# Each yearly release has its own dataset id. This one is data year 2024 (published April 2026); next year's
# id is on the dataset's page: data.cms.gov > Medicare Inpatient Hospitals - by Provider and Service > API.
INPATIENT_PROVIDER_SERVICE = "690ddc6c-2767-4618-b277-420ffb2bf27c"

def load_inpatient_drg(drg: str, dataset: str = INPATIENT_PROVIDER_SERVICE) -> pd.DataFrame:
    """Medicare Inpatient Hospitals by Provider and Service, one DRG, every hospital. Read as TEXT."""
    path = CACHE / f"cms_inpatient_{dataset[:8]}_drg{drg}.csv"
    if not path.exists():
        def fetch() -> bytes:
            url, rows, offset = f"https://data.cms.gov/data-api/v1/dataset/{dataset}/data", [], 0
            while True:
                chunk = json.loads(_get(url, {"filter[DRG_Cd]": drg, "size": 5000, "offset": offset}))
                rows += chunk
                if len(chunk) < 5000:
                    break
                offset += 5000
            return pd.DataFrame(rows).to_csv(index=False).encode()
        _fetch_or_mirror(path, fetch, f"cms/{path.name}")
    return pd.read_csv(path, dtype=str, keep_default_na=False)

raw = load_inpatient_drg("291")
hf = raw.rename(columns={"Rndrng_Prvdr_CCN": "ccn", "Rndrng_Prvdr_Org_Name": "hospital",
                         "Rndrng_Prvdr_State_Abrvtn": "state", "DRG_Desc": "drg_desc",
                         "Tot_Dschrgs": "discharges", "Avg_Submtd_Cvrd_Chrg": "charges",
                         "Avg_Tot_Pymt_Amt": "total_pay", "Avg_Mdcr_Pymt_Amt": "medicare_pay"})
hf = hf[["ccn", "hospital", "state", "drg_desc", "discharges", "charges", "total_pay", "medicare_pay"]].copy()
for c in ["discharges", "charges", "total_pay", "medicare_pay"]:
    hf[c] = to_num(hf[c])
print(hf.shape, "| DRG:", hf["drg_desc"].unique())
hf.head()


In [ ]:
# TODO: number of hospitals (is ccn unique?) and total discharges.
# TODO: 10th / 50th / 90th percentiles of total_pay and charges, and p90/p10 for each.
#       Hint: hf[["total_pay", "charges"]].quantile([0.1, 0.5, 0.9])


**Which varies more, and why:** *(edit this cell)*

## 2. Charges are not prices
The **charge-to-payment ratio** (`charges / total_pay`) says how many dollars a hospital bills for every dollar
Medicare actually pays. It is a measure of the chargemaster, not of what care costs.

**Baseline:** add the ratio as a column. Report its median, the share of hospitals that bill more than **3×**
what they are paid, and the five highest. Then find the hospitals where the ratio is **below 1** (paid more
than they billed). How many are there, and what kind of hospitals are they? (Read the names.)

In [ ]:
# TODO: hf["ratio"] = ...; median; share > 3; five highest (hospital, state, charges, total_pay, ratio)
# TODO: hospitals with ratio < 1: how many, and who are they?


**What the ratio tells you, and what is odd about the hospitals below 1:** *(edit this cell)*

## 3. Variation across states
Hospitals in the same state share a labor market, a payer mix and state law, so states are a natural first cut.

**Baseline:** for states with **at least 10 hospitals** in the file, compute the number of hospitals, the median
`total_pay`, the median `charges`, and the median `ratio`. Sort by median ratio and draw it as a horizontal bar
chart. Report the three lowest and three highest states, and where Connecticut falls. Then answer: why is
**Maryland** the lowest? *(Look up "Maryland all-payer model". One sentence is enough.)*

The medians here are medians of hospital averages, so every hospital counts once regardless of size. That is
the right choice for "what does a typical hospital do", the wrong one for "what does a typical patient see".

In [ ]:
import matplotlib.pyplot as plt

# TODO: by_state = hf.groupby("state").agg(hospitals=..., median_pay=..., median_charges=..., median_ratio=...)
# TODO: keep states with hospitals >= 10, sort by median_ratio, and plot:
# ax = by_state["median_ratio"].plot.barh(figsize=(6, 11), color="#4C72B0", width=0.7)
# ax.set_xlabel("median charges / total payment, DRG 291 (2024)"); ax.set_ylabel("")


**Lowest, highest, Connecticut, and why Maryland:** *(edit this cell)*

## 4. The price-transparency rule, and what a file looks like
Medicare data show what one public payer pays. What private insurers pay was a trade secret until the
**Hospital Price Transparency** rule, [45 CFR Part 180](https://www.ecfr.gov/current/title-45/subtitle-A/subchapter-E/part-180),
took effect on **January 1, 2021**. Every U.S. hospital must now publish:

- **One machine-readable file (MRF)** with its *standard charges* for **all** items and services
  ([§180.50](https://www.ecfr.gov/current/title-45/subtitle-A/subchapter-E/part-180/subpart-B/section-180.50)): the
  **gross charge** (the chargemaster price you saw in section 2), the **discounted cash price**, the
  **payer-specific negotiated charge** for each payer *and plan* by name, and the **de-identified minimum and
  maximum** negotiated charge. Since **July 1, 2024** the file must follow a CMS template (JSON or CSV) and data
  dictionary, published on [CMS's GitHub](https://github.com/CMSgov/hospital-price-transparency). It must be
  updated at least once a year and be downloadable without an account or a form.
- **A consumer-friendly list of at least 300 "shoppable" services** (§180.60), the ones a patient can plan ahead.
- **A way to find the file** (since January 1, 2024): a `.txt` file in the root of the website that lists each
  location's MRF URL, and a footer link labeled exactly "Price Transparency". CMS's
  [TXT generator](https://cmsgov.github.io/hpt-tool/txt-generator/) names it `cms-hpt.txt`. You will use it in the powerup.

CMS enforces the rule with warning notices, corrective action plans and civil monetary penalties
([CMS hospital price transparency](https://www.cms.gov/priorities/key-initiatives/hospital-price-transparency)).
The rule covers hospitals. Insurers have a parallel rule, *Transparency in Coverage*, with its own (much larger) files.

**Not every negotiated rate is a dollar amount.** A contract can say "80% of billed charges" or "the lesser of
$24,000 and 80% of charges". The file then carries a **percentage** or an **algorithm** instead of a dollar, plus the
median, 10th and 90th percentile of what the hospital was actually paid in the past 12–15 months.

The cell below downloads CMS's own **example file** for a fictional "West Mercy Hospital" (26 KB, public). Its
shape is: a list of items; each item has codes and one or more `standard_charges` (one per setting); each of those
has a list of `payers_information`, one entry per payer × plan.

**Baseline:** finish `flatten_mrf` so that it returns **one row per item × setting × payer × plan**. Run it on the
CMS example and report: how many rows, how many have **no dollar amount**, and which `methodology` values those
rows use.

In [ ]:
CMS_V3_EXAMPLE = ("https://raw.githubusercontent.com/CMSgov/hospital-price-transparency/master/"
                  "examples/JSON/v3_json_format_example.json")
cms_example = json.loads(_get(CMS_V3_EXAMPLE))
print(cms_example["hospital_name"], "| version", cms_example["version"],
      "|", len(cms_example["standard_charge_information"]), "items")
first = cms_example["standard_charge_information"][0]
print(json.dumps(first, indent=1)[:1200])   # look at one item before you flatten anything


In [ ]:
def flatten_mrf(doc: dict) -> pd.DataFrame:
    """CMS v3.0 JSON MRF -> one row per item x setting x payer x plan."""
    rows = []
    for item in doc["standard_charge_information"]:
        codes = "; ".join(f'{c["type"]} {c["code"]}' for c in item.get("code_information", []))
        for charge in item.get("standard_charges", []):
            for payer in charge.get("payers_information", []):
                # TODO: append one dict per payer x plan with these keys:
                #   description, codes, setting, gross_charge, discounted_cash   (from item / charge)
                #   payer_name, plan_name, methodology,
                #   negotiated_dollar (payer's "standard_charge_dollar"),
                #   negotiated_percentage ("standard_charge_percentage"), median_amount
                # Use .get(): a key that is missing must become None, not an error.
                pass
    return pd.DataFrame(rows)

ex = flatten_mrf(cms_example)
# TODO: rows; rows with no dollar amount; their methodology values


**Rows, rows without a dollar, and their methodologies:** *(edit this cell)*

## 5. Variation across payers, on a pre-cut sample
The CMS example has two payers, which is too few to measure variation. The cell below generates a **pre-cut
sample**: the same v3.0 shape, one hospital, 12 shoppable services and 11 payer-plan contracts, including a
Medicare Advantage plan and a Medicaid managed-care plan.

> **What this sample is.** It is **synthetic**: generated by the code below with a fixed seed, so everyone gets the
> same numbers. The *structure* is real (it is the CMS schema, with the messiness of real files: a contract that
> pays inpatient care as a percent of charges, and contracts that do not cover every service). The *numbers* are
> not: we chose how much each plan pays relative to a reference price. So it **can** teach you to flatten a file,
> to measure variation (max/min, IQR), and to compare negotiated rates with the gross charge and the cash price.
> It **cannot** tell you how much prices really vary, which insurer really pays more, or anything about a real
> hospital. For that you need a real file, which is the powerup.

**Baseline:** flatten the sample with your `flatten_mrf`, keep rows with a dollar amount, and build a table with
one row per service: number of payer-plans, min, 25th percentile, median, 75th percentile, max, **max/min**,
**IQR / median**, the gross charge and the discounted cash price. Report:
1. the service with the largest max/min, and its value;
2. max/min again with **commercial plans only** (drop "Medicare Advantage" and "Medicaid Managed Care"). How much of
   the spread was plan type?
3. for how many services the **cash price is below the median negotiated rate**, and what that means for a patient
   with a high-deductible plan;
4. how many rows had no dollar amount, and for which payer.

In [ ]:
import random

def make_sample_mrf(seed: int = 7) -> dict:
    """Pre-cut SYNTHETIC price-transparency file, CMS v3.0 JSON shape. Same seed, same file."""
    rng = random.Random(seed)
    services = [  # description, code, code type, setting, reference price ($), spread across payers
        ("Office visit, established patient, moderate", "99214", "CPT", "outpatient", 130, 0.10),
        ("Basic metabolic panel", "80048", "CPT", "outpatient", 10, 0.30),
        ("Chest X-ray, 2 views", "71046", "CPT", "outpatient", 60, 0.25),
        ("MRI brain without contrast", "70551", "CPT", "outpatient", 280, 0.35),
        ("CT abdomen and pelvis with contrast", "74177", "CPT", "outpatient", 330, 0.35),
        ("Emergency department visit, moderate", "99284", "CPT", "outpatient", 300, 0.20),
        ("Diagnostic colonoscopy", "45378", "CPT", "outpatient", 900, 0.20),
        ("Cataract removal with lens implant", "66984", "CPT", "outpatient", 2000, 0.15),
        ("Inguinal hernia repair", "49505", "CPT", "outpatient", 3200, 0.20),
        ("Vaginal delivery without complications", "807", "MS-DRG", "inpatient", 6500, 0.15),
        ("Heart failure and shock with MCC", "291", "MS-DRG", "inpatient", 11000, 0.15),
        ("Major joint replacement without MCC", "470", "MS-DRG", "inpatient", 14500, 0.15),
    ]
    contracts = [  # fictional payer, plan, typical multiple of the reference price
        ("Alder Health", "PPO", 2.4), ("Alder Health", "HMO", 1.9),
        ("Birch Mutual", "PPO", 2.9), ("Birch Mutual", "EPO", 2.5),
        ("Cedar Care", "Open Access", 1.7),
        ("Dogwood Insurance", "PPO", 2.1), ("Dogwood Insurance", "HMO", 1.6),
        ("Elm Community Plan", "Exchange Silver", 1.4),
        ("Fir National", "Choice", 3.3),
        ("Alder Health", "Medicare Advantage", 1.02),
        ("Cedar Care", "Medicaid Managed Care", 0.75),
    ]
    items = []
    for desc, code, ctype, setting, ref, spread in services:
        gross = round(ref * rng.uniform(3.5, 6.0), 2)
        payers = []
        for payer, plan, mult in contracts:
            if rng.random() < 0.06:                      # not every contract covers every service
                continue
            amount = round(ref * mult * rng.lognormvariate(0, spread), 2)
            p = {"payer_name": payer, "plan_name": plan}
            if setting == "inpatient" and payer == "Elm Community Plan":
                p.update(methodology="percent of total billed charges",   # no dollar amount in the file
                         standard_charge_percentage=round(100 * amount / gross), median_amount=amount,
                         count="1 through 10")
            else:
                p.update(methodology="case rate" if setting == "inpatient" else "fee schedule",
                         standard_charge_dollar=amount)
            payers.append(p)
        dollars = [p["standard_charge_dollar"] for p in payers if "standard_charge_dollar" in p]
        items.append({"description": desc, "code_information": [{"code": code, "type": ctype}],
                      "standard_charges": [{"setting": setting, "gross_charge": gross,
                                            "discounted_cash": round(gross * rng.uniform(0.35, 0.6), 2),
                                            "minimum": min(dollars), "maximum": max(dollars),
                                            "payers_information": payers}]})
    return {"hospital_name": "Sample Regional Medical Center (synthetic)", "version": "3.0.0",
            "last_updated_on": "2026-07-01", "standard_charge_information": items}

sample = make_sample_mrf()
rates = flatten_mrf(sample)
print(rates.shape)
rates.head()


In [ ]:
# TODO: d = rates with a dollar amount
# TODO: per service (groupby "description"): n, min, p25, median, p75, max; then max_min and iqr_over_median;
#       gross_charge and discounted_cash ("first" works: they are the same on every row of a service)
# TODO: the same max/min for commercial plans only
# TODO: cash below median: how many services?  rows with no dollar: how many, which payer?


In [ ]:
# Each dot is one payer-plan's rate, divided by that service's median rate, so services of very
# different prices share one axis. Run as is once your table works.
d = rates.dropna(subset=["negotiated_dollar"]).copy()
d["rel"] = d["negotiated_dollar"] / d.groupby("description")["negotiated_dollar"].transform("median")
order = d.groupby("description")["rel"].agg(lambda s: s.max() / s.min()).sort_values().index
fig, ax = plt.subplots(figsize=(8, 5.5))
for i, svc in enumerate(order):
    pts = d[d["description"].eq(svc)]
    gov = pts["plan_name"].isin(["Medicare Advantage", "Medicaid Managed Care"])
    ax.scatter(pts.loc[~gov, "rel"], [i] * (~gov).sum(), s=40, color="#4C72B0", label="commercial" if i == 0 else None)
    ax.scatter(pts.loc[gov, "rel"], [i] * gov.sum(), s=40, color="#DD8452", marker="D",
               label="Medicare Advantage / Medicaid" if i == 0 else None)
ax.axvline(1, color="#555555", linewidth=1, linestyle="--")
ax.set_yticks(range(len(order)), order); ax.set_xlabel("negotiated rate / median rate for the service")
ax.set_title("Same hospital, same service, different payer (synthetic sample)", loc="left")
ax.legend(loc="lower right", frameon=False); plt.tight_layout(); plt.show()


**Your four answers:** *(edit this cell)*

**One question about real prices this sample cannot answer, and the data you would need:** *(edit this cell)*

## 6. Powerups
**(a) A real file, with Polars streaming.** Real MRFs are big: hundreds of megabytes to tens of gigabytes, more than
pandas can hold in memory. Polars can *scan* a file lazily and *stream* it in chunks, so only the rows you filter
for are ever held in memory.
1. **Find the file.** Pick a hospital and open `https://<hospital website>/cms-hpt.txt` in your browser (the rule
   requires it in the website's root; see section 4). It lists `location-name`, `source-page-url`, `mrf-url`
   and a contact for each location. `read_cms_hpt()` below parses one. Some hospital systems list all their
   hospitals in one file.
2. **Download it.** `download()` streams to disk without holding the file in memory. Some sites block scripts
   (a "Just a moment..." page is a Cloudflare bot check): download in your browser instead and upload it with Colab's **Files** panel (folder icon, left sidebar) into the `data` folder, and set `MRF_PATH` to it. Uploads are deleted when the runtime disconnects; for a big file, mount Drive (Files panel → Drive icon) and read it from `/content/drive/MyDrive/`. Unzip `.zip` files first.
3. **Scan it.** For a **CSV** MRF (most hospitals publish the CMS "tall" CSV), rows 1–2 are hospital metadata and
   the real header is row 3, hence `skip_rows=2`. Read everything as text (`infer_schema=False`), filter to one
   code, then `collect(engine="streaming")`. Until you have a file, the cell runs on CMS's public tall-CSV example.
   For a **JSON** MRF, Polars cannot stream one giant nested document: stream it with `ijson`
   (`ijson.items(f, "standard_charge_information.item")`), feed each item to your `flatten_mrf` logic in batches,
   write the batches to Parquet, and `pl.scan_parquet` the result.
4. **Report** your hospital, the file's size and format, and the max/min across payers for one service.

**(b) The same DRG in both worlds.** In your hospital's MRF, find MS-DRG **291**. Divide each commercial
negotiated rate by that hospital's `total_pay` for DRG 291 from section 1 (match by name and city). This
"percent of Medicare" is how researchers compare commercial prices across hospitals. What is your hospital's,
and why is matching the two files harder than it sounds?

In [ ]:
# --- Week 7 powerup packages: polars (and ijson for JSON files) --------------------
import importlib.util, shutil, subprocess, sys

_need = [pkg for pkg, mod in [("polars", "polars"), ("ijson", "ijson")] if importlib.util.find_spec(mod) is None]
if _need:
    if shutil.which("uv"):  # Hex: the kernel's virtualenv has no pip module, but uv is on the PATH
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_need], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=True)
    importlib.invalidate_caches()
print("week 7 powerup packages ready")


In [ ]:
import polars as pl

def read_cms_hpt(site: str) -> pd.DataFrame:
    """Parse https://<site>/cms-hpt.txt into one row per hospital location."""
    text = _get(f"https://{site}/cms-hpt.txt", timeout=30).decode("utf-8", "replace")
    rows, row = [], {}
    for line in text.splitlines() + [""]:
        if ":" in line:
            k, v = line.split(":", 1); row[k.strip()] = v.strip()
        elif row:
            rows.append(row); row = {}
    return pd.DataFrame(rows)

def download(url: str, path: Path) -> Path:
    """Stream a large file to disk in 1 MB chunks."""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (healthcare-analytics-course)"})
    with urllib.request.urlopen(req, timeout=600) as r, open(path, "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    return path

# TODO: hpt = read_cms_hpt("www.<your hospital>.org"); hpt   -> copy the mrf-url you want
MRF_PATH = CACHE / "my_hospital_mrf.csv"          # TODO: your downloaded / uploaded file
if not MRF_PATH.exists():                          # stand-in until you have one: CMS's public tall-CSV example
    download("https://raw.githubusercontent.com/CMSgov/hospital-price-transparency/master/examples/CSV/"
             "Tall%20Format%20Examples/V3.0.0_Tall_CSV_Format_Example.csv", MRF_PATH)

lf = pl.scan_csv(MRF_PATH, skip_rows=2, infer_schema=False)          # lazy: nothing is read yet
lf = lf.rename({c: c.replace(" ", "") for c in lf.collect_schema().names()})   # "code | 1" -> "code|1"
code_cols = [c for c in lf.collect_schema().names() if c.startswith("code|") and not c.endswith("|type")]
print(MRF_PATH, round(MRF_PATH.stat().st_size / 1e6, 1), "MB | code columns:", code_cols)
# TODO: filter rows where any code column equals your code (pl.any_horizontal), cast
#       "standard_charge|negotiated_dollar" to Float64 (strict=False), and collect(engine="streaming").
